# GPR-URBAN-VN: Getting Started and Data Loader Walkthrough

This interactive notebook demonstrates how to explore and use the **GPR-URBAN-VN** benchmark dataset archived on **Harvard Dataverse** (`doi:10.7910/DVN/VYMICL`).

### Dataset Highlights:
* **10,000 independent urban pavement scenarios** (5,000 at 400 MHz and 5,000 at 700 MHz, unpaired).
* **19,000 buried targets across 7 classes** with exact physical geometry and moisture variations.
* **Three Tiers of Data:**
  1. `tier1_scene_metadata.parquet`: Scene stratigraphy, layer dielectric parameters, and target lists.
  2. `part_xxx_bscans.tar.gz`: Full-precision float32 electric wavefields $E_z(t, x)$.
  3. `part_xxx_images.tar.gz` and `targets.csv`: $640\times 640$ 8-bit B-scans with 17,758 verified bounding boxes.

In [ ]:
# Install or import required standard libraries
import os
import json
import io
import tarfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

print('Libraries imported successfully!')

## 1. Inspecting Tier-1 Scene Metadata

`tier1_scene_metadata.parquet` contains dielectric and geometric parameters for all 10,000 scenarios sampled via an 11-dimensional Latin hypercube design.

In [ ]:
# Load Tier-1 metadata
metadata_path = 'tier1_scene_metadata.parquet'
if os.path.exists(metadata_path):
    df_meta = pd.read_parquet(metadata_path)
    print(f'Total scenarios loaded: {len(df_meta):,}')
    display(df_meta[['sample_id', 'frequency_mhz', 'er_asphalt', 'er_base', 'er_subgrade', 'roughness_std_m']].head())
else:
    print(f'Metadata file not found at {metadata_path}. Please place it in the working directory.')

## 2. Inspecting Per-Target Labels and Physical Ground Truth

`targets.csv` lists all 19,000 targets, their exact burial depths, dimensions, and the `visible` flag indicating whether the reflection pulse arrives within the survey recording window.

In [ ]:
# Load targets table
targets_path = 'targets.csv'
if os.path.exists(targets_path):
    df_targets = pd.read_csv(targets_path)
    print(f'Total targets: {len(df_targets):,}')
    print(f'Visible (boxed) targets: {(df_targets["visible"] == 1).sum():,}')
    print(f'Invisible targets: {(df_targets["visible"] == 0).sum():,}')
    
    print('\nTarget distribution by class:')
    print(df_targets['class_name'].value_counts())
else:
    print(f'targets.csv not found at {targets_path}.')

## 3. Loading Float32 Wavefields from Tier-2 Archives

Each `part_xxx_bscans.tar.gz` archive contains 100 scenario wavefields stored as `.npz` files with raw electric field arrays $E_z(t, x)$ and AGC-normalized versions.

In [ ]:
# Inspect a sample wavefield archive if available
archive_path = 'part_000_bscans.tar.gz'
if os.path.exists(archive_path):
    with tarfile.open(archive_path, 'r:gz') as tar:
        npz_member = next(m for m in tar if m.name.endswith('.npz'))
        data_bytes = tar.extractfile(npz_member).read()
        z = np.load(io.BytesIO(data_bytes))
        
        raw_wavefield = z['bscan_raw']
        time_s = z['time_s']
        tx_x_m = z['tx_x_m']
        
        print(f'Loaded wavefield: {npz_member.name}')
        print(f'Array shape: {raw_wavefield.shape} (time samples x receiver traces)')
        
        # Visualize wavefield
        plt.figure(figsize=(6, 4), dpi=150)
        vmax = np.percentile(np.abs(raw_wavefield), 99.5)
        plt.imshow(raw_wavefield, aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
                   extent=[tx_x_m[0], tx_x_m[-1], time_s[-1]*1e9, time_s[0]*1e9])
        plt.xlabel('Antenna Position (m)')
        plt.ylabel('Two-Way Travel Time (ns)')
        plt.title(f'Raw Unclipped Wavefield ({z["sample_id"]})')
        plt.colorbar(label='Electric Field $E_z$ (V/m)')
        plt.tight_layout()
        plt.show()
else:
    print('To run this cell, download part_000_bscans.tar.gz from Harvard Dataverse.')

## 4. Visualizing B-Scans with Bounding Box Annotations (Tier 3)

Each B-scan image in `part_xxx_images.tar.gz` is a $640\times 640$ 8-bit PNG paired with standard YOLO formatted `.txt` labels.

In [ ]:
# Demonstration: plot a sample B-scan with bounding box overlay
img_archive = 'part_000_images.tar.gz'
if os.path.exists(img_archive):
    with tarfile.open(img_archive, 'r:gz') as tar:
        png_member = next(m for m in tar if m.name.endswith('.png'))
        txt_member = next(m for m in tar if m.name.endswith('.txt') and m.name[:-4] == png_member.name[:-4])
        
        img = Image.open(io.BytesIO(tar.extractfile(png_member).read())).convert('RGB')
        labels = tar.extractfile(txt_member).read().decode('utf-8').strip().split('\n')
        
        CLASSES = ['pec', 'upvc_air', 'upvc_water', 'upvc_triphase', 'cable_bundle', 'box_culvert', 'sinkhole_void']
        
        fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
        ax.imshow(img, cmap='gray')
        w, h = img.size
        
        for line in labels:
            if not line.strip(): continue
            cls_id, cx, cy, bw, bh = map(float, line.split())
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            rect = patches.Rectangle((x1, y1), bw * w, bh * h, linewidth=1.8, edgecolor='#D55E00', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1 + 3, max(12, y1 - 4), CLASSES[int(cls_id)], color='#D55E00', fontsize=8, fontweight='bold')
            
        ax.set_title(f'B-scan Image and YOLO Labels ({png_member.name})')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('To run this cell, download part_000_images.tar.gz from Harvard Dataverse.')

## 5. Splitting and Training Preparation

The official leak-free split `split_gpr_urban_vn_10k_base.json` groups scenarios by exact geometry ($1\,\mu\text{m}$ tolerance):
* **7,000 Training Scenarios** (3,500 at 400 MHz, 3,500 at 700 MHz)
* **1,000 Validation Scenarios** (500 at 400 MHz, 500 at 700 MHz)
* **2,000 Test Scenarios** (1,000 at 400 MHz, 1,000 at 700 MHz)

Benchmark scripts and detector models (YOLO11, RT-DETR) are open-source on GitHub:
👉 **Repository:** [https://github.com/BonTapHoa/gpr-urban-vn-benchmark](https://github.com/BonTapHoa/gpr-urban-vn-benchmark)